# PART2 Colab GPU Runner (Legacy)

This notebook is kept as a legacy Colab helper, not as the canonical source for the report numbers.

For report reproduction, use `scripts/rerun_part2_report_20260611.sh` and `docs/part2/05_report_rerun_protocol_20260611.md` on the GCP GPU VM. The report-facing summary is `results/report_summary_20260611.csv`.

The older Colab flow used different defaults and checkpoint naming, so do not use its output in the report unless the cells are manually updated to match the canonical rerun protocol.

In [ ]:
RUN_LEGACY_COLAB = False
if not RUN_LEGACY_COLAB:
    raise RuntimeError(
        'This notebook is legacy and not the canonical report rerun. '
        'Set RUN_LEGACY_COLAB = True only for a separate exploratory Colab run.'
    )

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
CODE=/content/NLP_Study_PART2_CODE
OUT=/content/drive/MyDrive/NLP_Study_PART2_OUTPUT
rm -rf "$CODE"
git clone -b part2 --single-branch https://github.com/Y0onSe0/NLP_Study.git "$CODE"
mkdir -p "$OUT/checkpoints" "$OUT/predictions" "$OUT/results" "$OUT/logs"
if [ -f "$OUT/results/paraphrase_experiments.csv" ]; then
  cp "$OUT/results/paraphrase_experiments.csv" "$OUT/results/paraphrase_experiments.backup_$(date +%Y%m%d_%H%M%S).csv"
  rm "$OUT/results/paraphrase_experiments.csv"
fi
cat > /content/part2_paths.env <<EOF
export CODE=$CODE
export OUT=$OUT
EOF
cd "$CODE"
git branch --show-current
git log -1 --oneline
git status --short
echo "Outputs: $OUT"

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
cd "$CODE"
python -m pip install -q \
  transformers==4.46.3 \
  tokenizers==0.20.0 \
  einops==0.8.0 \
  sacrebleu==2.5.1 \
  tqdm==4.58.0 \
  scikit-learn \
  pandas
python - <<'PY'
import torch, transformers, tokenizers, sklearn, pandas
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('tokenizers', tokenizers.__version__)
print('sklearn', sklearn.__version__)
print('pandas', pandas.__version__)
print('cuda', torch.cuda.is_available())
PY

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Use Runtime > Change runtime type > GPU.'
print(torch.cuda.get_device_name(0))

## 1. Smoke Test

This validates the environment with a tiny GPU run.

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
cd "$CODE"
python paraphrase_detection.py \
  --use_gpu \
  --mode train_dev \
  --epochs 1 \
  --batch_size 2 \
  --max_train_examples 128 \
  --max_dev_examples 128 \
  --prompt_template baseline \
  --output_tag smoke-baseline \
  --filepath "$OUT/checkpoints/smoke-baseline.pt" \
  --para_dev_out "$OUT/predictions/para-dev-smoke-baseline.csv" \
  --experiment_log "$OUT/results/paraphrase_experiments.csv" \
  2>&1 | tee "$OUT/logs/smoke-baseline.log"

## 2. Prompt Screening

Runs `baseline`, `direct`, and `meaning` on the same subset.

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
cd "$CODE"
for PROMPT in baseline direct meaning; do
  python paraphrase_detection.py \
    --use_gpu \
    --mode train_dev \
    --epochs 1 \
    --batch_size 8 \
    --max_train_examples 20000 \
    --max_dev_examples 5000 \
    --prompt_template "$PROMPT" \
    --output_tag "screen-$PROMPT" \
    --filepath "$OUT/checkpoints/screen-$PROMPT.pt" \
    --para_dev_out "$OUT/predictions/para-dev-screen-$PROMPT.csv" \
    --experiment_log "$OUT/results/paraphrase_experiments.csv" \
    2>&1 | tee "$OUT/logs/screen-$PROMPT.log"
done

In [ ]:
import os
import pandas as pd

OUT = '/content/drive/MyDrive/NLP_Study_PART2_OUTPUT'
log_path = f'{OUT}/results/paraphrase_experiments.csv'
df = pd.read_csv(log_path)
screen = df[df['output_tag'].astype(str).str.startswith('screen-')].copy()
screen['dev_acc_num'] = pd.to_numeric(screen['dev_acc'], errors='coerce')
screen = screen.dropna(subset=['dev_acc_num']).sort_values('dev_acc_num', ascending=False)
if screen.empty:
    raise RuntimeError('No prompt-screening rows found in paraphrase_experiments.csv')
best_prompt = str(screen.iloc[0]['prompt_template'])
with open('/content/part2_run.env', 'w') as f:
    f.write(f'export BEST_PROMPT={best_prompt}\n')
    f.write('export DEV_SELECTED_THRESHOLD=0.50\n')
print('BEST_PROMPT =', best_prompt)
display(screen[['output_tag', 'prompt_template', 'dev_acc', 'checkpoint', 'para_dev_out']])

## 3. Full Training

Uses the automatically selected prompt and saves the best checkpoint to Drive.

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
source /content/part2_run.env
cd "$CODE"
python paraphrase_detection.py \
  --use_gpu \
  --mode train_dev \
  --epochs 10 \
  --batch_size 8 \
  --lr 1e-5 \
  --prompt_template "$BEST_PROMPT" \
  --output_tag prompt-full \
  --filepath "$OUT/checkpoints/prompt-full.pt" \
  --para_dev_out "$OUT/predictions/para-dev-prompt-full.csv" \
  --experiment_log "$OUT/results/paraphrase_experiments.csv" \
  2>&1 | tee "$OUT/logs/prompt-full.log"
test -f "$OUT/checkpoints/prompt-full.pt" && echo checkpoint-ok

## 4. Bidirectional Dev Prediction

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
source /content/part2_run.env
cd "$CODE"
python paraphrase_detection.py \
  --use_gpu \
  --mode dev_predict \
  --filepath "$OUT/checkpoints/prompt-full.pt" \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold 0.5 \
  --output_tag prompt-bidir-dev \
  --para_dev_out "$OUT/predictions/para-dev-prompt-bidir-dev.csv" \
  --experiment_log "$OUT/results/paraphrase_experiments.csv" \
  2>&1 | tee "$OUT/logs/prompt-bidir-dev.log"

## 5. Threshold Calibration

The next Python cell reads the selected threshold automatically.

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
source /content/part2_run.env
cd "$CODE"
python paraphrase_detection.py \
  --use_gpu \
  --mode calibrate_dev \
  --filepath "$OUT/checkpoints/prompt-full.pt" \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold_min 0.30 \
  --threshold_max 0.70 \
  --threshold_step 0.01 \
  --output_tag prompt-bidir-calib \
  --experiment_log "$OUT/results/paraphrase_experiments.csv" \
  2>&1 | tee "$OUT/logs/prompt-bidir-calib.log"

In [ ]:
import os
import pandas as pd

OUT = '/content/drive/MyDrive/NLP_Study_PART2_OUTPUT'
df = pd.read_csv(f'{OUT}/results/paraphrase_experiments.csv')
calib = df[df['output_tag'].astype(str).eq('prompt-bidir-calib')].copy()
calib['selected_threshold_num'] = pd.to_numeric(calib['selected_threshold'], errors='coerce')
calib = calib.dropna(subset=['selected_threshold_num'])
if calib.empty:
    raise RuntimeError('No calibrated threshold found in paraphrase_experiments.csv')
threshold = float(calib.iloc[-1]['selected_threshold_num'])
with open('/content/part2_run.env') as f:
    lines = [line for line in f.read().splitlines() if not line.startswith('export DEV_SELECTED_THRESHOLD=')]
lines.append(f'export DEV_SELECTED_THRESHOLD={threshold:.4f}')
with open('/content/part2_run.env', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print('DEV_SELECTED_THRESHOLD =', f'{threshold:.4f}')
display(df.tail(20))

## 6. Error Analysis

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
source /content/part2_run.env
cd "$CODE"
python paraphrase_detection.py \
  --use_gpu \
  --mode error_analysis \
  --filepath "$OUT/checkpoints/prompt-full.pt" \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold "$DEV_SELECTED_THRESHOLD" \
  --output_tag prompt-bidir-error \
  --experiment_log "$OUT/results/paraphrase_experiments.csv" \
  --error_analysis_out "$OUT/results/error_analysis_para.csv" \
  2>&1 | tee "$OUT/logs/prompt-bidir-error.log"

## 7. Final Test Prediction

This uses the dev-selected prompt and threshold.

In [ ]:
%%bash
set -euo pipefail
source /content/part2_paths.env
source /content/part2_run.env
cd "$CODE"
python paraphrase_detection.py \
  --use_gpu \
  --mode test_predict \
  --filepath "$OUT/checkpoints/prompt-full.pt" \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold "$DEV_SELECTED_THRESHOLD" \
  --para_test_out "$OUT/predictions/para-test-final.csv" \
  --output_tag final-test \
  --experiment_log "$OUT/results/paraphrase_experiments.csv" \
  2>&1 | tee "$OUT/logs/final-test.log"
cp "$OUT/predictions/para-test-final.csv" predictions/para-test-final.csv
python prepare_submit.py
cp nlp2025-1_project_outputs.zip "$OUT/"
wc -l "$OUT/predictions/para-test-final.csv"
head -n 5 "$OUT/predictions/para-test-final.csv"
tail -n 10 "$OUT/results/paraphrase_experiments.csv"
echo "BEST_PROMPT=$BEST_PROMPT"
echo "DEV_SELECTED_THRESHOLD=$DEV_SELECTED_THRESHOLD"
echo "Saved outputs to $OUT"